Import Library

In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from lime.lime_tabular import LimeTabularExplainer


Loading Dataset

In [55]:
dd = pd.read_csv('diabetic_data.csv')

Data Structure Check

In [56]:
print(dd.info())
print(dd.head())
print(dd.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

Handling Missing Values

In [57]:
# Check missing values
missing = dd.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

# Drop columns with too many missing or irrelevant values
dd.drop(['weight', 'payer_code', 'medical_specialty'], axis=1, inplace=True)

# Drop duplicates if any
dd.drop_duplicates(inplace=True)

# Check class distribution
dd['readmitted'].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Data Transformation

In [58]:
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]
medication_map = {
    'No': 0,
    'Down': 1,
    'Steady': 2,
    'Up': 3
}
for col in medication_cols:
    if col in dd.columns:
        dd.loc[:, col] = dd[col].map(medication_map)


Data Standardization

In [59]:
#converting the 10-year interval into averaged value
age_map = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}
dd.loc[:, 'age'] = dd['age'].map(age_map)

def group_icd(code):
    try:
        code = float(code)
        if 390 <= code <= 459 or code == 785:
            return 'circulatory'
        elif 460 <= code <= 519 or code == 786:
            return 'respiratory'
        elif 520 <= code <= 579 or code == 787:
            return 'digestive'
        elif code == 250:
            return 'diabetes'
        elif 800 <= code <= 999:
            return 'injury'
        else:
            return 'other'
    except:
        return 'unknown'


Feature Engineering

In [60]:
# Simplify 'readmitted' to binary target
dd['readmitted'] = dd['readmitted'].apply(lambda x: 0 if x == 'NO' else 1)

# Encode categorical features
cat_cols = dd.select_dtypes(include='object').columns

le = LabelEncoder()
for col in cat_cols:
    dd[col] = le.fit_transform(dd[col].astype(str))

#Create New Column 'Total Visits'
dd['total_visits'] = (
    dd['number_outpatient'] +
    dd['number_emergency'] +
    dd['number_inpatient']
)

dd['num_medications_used'] = (dd[medication_cols] != 'No').sum(axis=1)
dd['long_stay'] = (dd['time_in_hospital'] > 7).astype(int)
dd['multi_morbidity'] = (dd['number_diagnoses'] >= 8).astype(int)
dd['abnormal_glucose'] = dd['max_glu_serum'].isin(['>200', '>300']).astype(int)
dd['abnormal_A1C'] = dd['A1Cresult'].isin(['>7', '>8']).astype(int)

dd.to_csv("diabetic_data_processed.csv", index=False)


Feature Selection

In [61]:
dd.drop(['encounter_id', 'patient_nbr'], axis=1, inplace=True)

Data Splitting

In [62]:
# Split into features and target
X = dd.drop('readmitted', axis=1)
y = dd['readmitted']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Sampling

In [63]:
# If your column name is exactly "readmitted"
dd['readmitted'].value_counts()

print("Before undersampling:", Counter(y_train))

# Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("After undersampling:", Counter(y_train_under))


Before undersampling: Counter({0: 43912, 1: 37500})
After undersampling: Counter({0: 37500, 1: 37500})


In [64]:
# Train a RandomForest model with class weights
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# Predict
y_pred = rf.predict(X_test)

# Evaluation
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Random Forest Accuracy: 0.6367298811044512
[[8164 2788]
 [4606 4796]]
              precision    recall  f1-score   support

           0       0.64      0.75      0.69     10952
           1       0.63      0.51      0.56      9402

    accuracy                           0.64     20354
   macro avg       0.64      0.63      0.63     20354
weighted avg       0.64      0.64      0.63     20354



XGBoost

In [65]:
# Train XGBoost model
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)

# Prediction and evaluation
y_pred_xgb = xgb.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

c:\Users\HP PAVILION 15\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:03:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.6454259604991648
[[7801 3151]
 [4066 5336]]
              precision    recall  f1-score   support

           0       0.66      0.71      0.68     10952
           1       0.63      0.57      0.60      9402

    accuracy                           0.65     20354
   macro avg       0.64      0.64      0.64     20354
weighted avg       0.64      0.65      0.64     20354



LightGBM

In [66]:
lgb_model = lgb.LGBMClassifier(random_state=42)
lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)

print("=== LightGBM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))
print(classification_report(y_test, y_pred_lgb))


[LightGBM] [Info] Number of positive: 37500, number of negative: 43912
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008175 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1157
[LightGBM] [Info] Number of data points in the train set: 81412, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.460620 -> initscore=-0.157847
[LightGBM] [Info] Start training from score -0.157847
=== LightGBM ===
Accuracy: 0.6497985653925519
[[8032 2920]
 [4208 5194]]
              precision    recall  f1-score   support

           0       0.66      0.73      0.69     10952
           1       0.64      0.55      0.59      9402

    accuracy                           0.65     20354
   macro avg       0.65      0.64      0.64     20354
weighted avg       0.65      0.65      0.65     20354



CatBoost

In [67]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=0,
    random_seed=42
)
cat_model.fit(X_train, y_train)

y_pred_cat = cat_model.predict(X_test)

print("=== CatBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_cat))
print(confusion_matrix(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat))


=== CatBoost ===
Accuracy: 0.6511250859781861
[[8049 2903]
 [4198 5204]]
              precision    recall  f1-score   support

           0       0.66      0.73      0.69     10952
           1       0.64      0.55      0.59      9402

    accuracy                           0.65     20354
   macro avg       0.65      0.64      0.64     20354
weighted avg       0.65      0.65      0.65     20354



### Model Tuning - Random Search

Random Forest

In [68]:
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

param_dist_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2', None]
}

rand_search_rf = RandomizedSearchCV(
    rf, param_distributions=param_dist_rf, n_iter=20, cv=3,
    scoring='f1', verbose=2, random_state=42
)
rand_search_rf.fit(X_train, y_train)
y_pred_rf = rand_search_rf.predict(X_test)

print("=== Random Forest ===")
print("Best Params:", rand_search_rf.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END max_depth=30, max_features=None, min_samples_leaf=5, min_samples_split=5, n_estimators=100; total time=  40.0s
[CV] END max_depth=30, max_features=None, min_samples_leaf=5, min_samples_split=5, n_estimators=100; total time=  40.2s
[CV] END max_depth=30, max_features=None, min_samples_leaf=5, min_samples_split=5, n_estimators=100; total time=  37.1s
[CV] END max_depth=None, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time= 4.3min
[CV] END max_depth=None, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time= 7.7min
[CV] END max_depth=None, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=500; total time= 7.9min
[CV] END max_depth=10, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  23.0s
[CV] END max_depth=10, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimato

XGBoost

In [69]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, scale_pos_weight=1.0)

param_dist_xgb = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

rand_search_xgb = RandomizedSearchCV(
    xgb, param_distributions=param_dist_xgb, n_iter=20, cv=3,
    scoring='f1', verbose=2, random_state=42
)
rand_search_xgb.fit(X_train, y_train)
y_pred_xgb = rand_search_xgb.predict(X_test)

print("=== XGBoost ===")
print("Best Params:", rand_search_xgb.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\HP PAVILION 15\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:16:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=9, n_estimators=300, subsample=0.6; total time=   1.2s


c:\Users\HP PAVILION 15\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:16:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=9, n_estimators=300, subsample=0.6; total time=   1.1s


...

[CV] END colsample_bytree=0.6, learning_rate=0.2, max_depth=3, n_estimators=400, subsample=0.6; total time=   0.5s


c:\Users\HP PAVILION 15\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:16:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END colsample_bytree=0.6, learning_rate=0.2, max_depth=3, n_estimators=400, subsample=0.6; total time=   0.5s


c:\Users\HP PAVILION 15\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:16:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
Best Params: {'subsample': 0.8, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
Accuracy: 0.6507811732337624
[[7918 3034]
 [4074 5328]]
              precision    recall  f1-score   support

           0       0.66      0.72      0.69     10952
           1       0.64      0.57      0.60      9402

    accuracy                           0.65     20354
   macro avg       0.65      0.64      0.65     20354
weighted avg       0.65      0.65      0.65     20354



LightGBM

In [70]:
lgb_model = lgb.LGBMClassifier(random_state=42)

param_dist_lgb = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [5, 10, 15, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 50, 70],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

rand_search_lgb = RandomizedSearchCV(
    lgb_model, param_distributions=param_dist_lgb, n_iter=20, cv=3,
    scoring='f1', verbose=2, random_state=42
)
rand_search_lgb.fit(X_train, y_train)
y_pred_lgb = rand_search_lgb.predict(X_test)

print("=== LightGBM ===")
print("Best Params:", rand_search_lgb.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))
print(classification_report(y_test, y_pred_lgb))


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 25000, number of negative: 29274
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003414 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1135
[LightGBM] [Info] Number of data points in the train set: 54274, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.460626 -> initscore=-0.157824
[LightGBM] [Info] Start training from score -0.157824
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=10, n_estimators=400, num_leaves=50, subsample=1.0; total time=   1.2s
[LightGBM] [Info] Number of positive: 25000, number of negative: 29275
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003291 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is

CatBoost

In [71]:
cat_model = CatBoostClassifier(verbose=0, random_seed=42)

param_dist_cat = {
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'iterations': [200, 500, 1000],
    'l2_leaf_reg': [3, 5, 7, 9],
    'border_count': [32, 64, 128],
    'bagging_temperature': [0, 0.5, 1]
}

rand_search_cat = RandomizedSearchCV(
    cat_model, param_distributions=param_dist_cat, n_iter=20, cv=3,
    scoring='f1', verbose=2, random_state=42
)
rand_search_cat.fit(X_train, y_train)
y_pred_cat = rand_search_cat.predict(X_test)

print("=== CatBoost ===")
print("Best Params:", rand_search_cat.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_cat))
print(confusion_matrix(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat))


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END bagging_temperature=0.5, border_count=32, depth=4, iterations=500, l2_leaf_reg=5, learning_rate=0.2; total time=  22.0s
[CV] END bagging_temperature=0.5, border_count=32, depth=4, iterations=500, l2_leaf_reg=5, learning_rate=0.2; total time=  21.7s
[CV] END bagging_temperature=0.5, border_count=32, depth=4, iterations=500, l2_leaf_reg=5, learning_rate=0.2; total time=  21.6s
[CV] END bagging_temperature=1, border_count=32, depth=6, iterations=200, l2_leaf_reg=3, learning_rate=0.05; total time=   8.0s
[CV] END bagging_temperature=1, border_count=32, depth=6, iterations=200, l2_leaf_reg=3, learning_rate=0.05; total time=   8.8s
[CV] END bagging_temperature=1, border_count=32, depth=6, iterations=200, l2_leaf_reg=3, learning_rate=0.05; total time=   9.3s
[CV] END bagging_temperature=0.5, border_count=32, depth=6, iterations=200, l2_leaf_reg=5, learning_rate=0.01; total time=   9.3s
[CV] END bagging_temperature=0.5, bord

KeyboardInterrupt: 

XAI (LIME + Catboost)

In [ ]:

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=7,
    border_count=64,
    bagging_temperature=1,
    verbose=100,
    random_state=42
)
cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_test)
explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns.tolist(),
    class_names=['No_Readmit', 'Readmit'],
    mode='classification'
)
patient_index = 0
patient_data = X_test.iloc[patient_index].values

exp = explainer.explain_instance(
    data_row=patient_data,
    predict_fn=cat_model.predict_proba,
    num_features=5  # top 10 contributing features
)
output_file = f"lime_patient_{patient_index}_explanation.html"
exp.save_to_file(output_file)
print(f"LIME explanation saved to: {output_file}")

0:	learn: 0.6875931	total: 31.4ms	remaining: 31.4s
100:	learn: 0.6184351	total: 3.96s	remaining: 35.2s
200:	learn: 0.6060174	total: 7.94s	remaining: 31.6s
300:	learn: 0.5949773	total: 11.9s	remaining: 27.6s
400:	learn: 0.5849280	total: 15.9s	remaining: 23.8s
500:	learn: 0.5753819	total: 19.9s	remaining: 19.8s
600:	learn: 0.5667613	total: 23.7s	remaining: 15.7s
700:	learn: 0.5586877	total: 27.5s	remaining: 11.7s
800:	learn: 0.5508659	total: 31s	remaining: 7.69s
900:	learn: 0.5442801	total: 35s	remaining: 3.84s
999:	learn: 0.5376165	total: 38.5s	remaining: 0us
LIME explanation saved to: lime_patient_0_explanation.html


In [ ]:
import joblib

joblib.dump(cat_model, "catboost_readmission_model.pkl")


['catboost_readmission_model.pkl']

In [ ]:
joblib.dump(list(X_train.columns), "feature_names.pkl")

['feature_names.pkl']